###Now spliting the data chronnlogically

In [3]:

from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path(r"C:\my Files\RiskPredANDMarketEvaluation")

DATA_PATH = (
    PROJECT_ROOT
    / "experiments"
    / "outputs"
    / "model_ready_candidates.csv"
)

OUTPUT_DIR = PROJECT_ROOT / "experiments" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET = "stress_event_5pct_10d"

FEATURES = [
    "return_1d",
    "return_5d",
    "return_10d",
    "volatility_5d",
    "volatility_20d",
    "drawdown",
    "volume_change_1d",
    "volume_ratio_20d",
]

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH}")

data = pd.read_csv(DATA_PATH, parse_dates=["date"])
data = data.sort_values(["date", "ticker"]).reset_index(drop=True)

print("Dataset shape:", data.shape)
print("Tickers:", sorted(data["ticker"].unique()))
print("Date range:", data["date"].min(), "to", data["date"].max())
print("\nTarget distribution:")
print(data[TARGET].value_counts())


Dataset shape: (4896, 11)
Tickers: ['AMD', 'INTC', 'NVDA', 'TSM']
Date range: 2021-11-12 00:00:00 to 2026-09-30 00:00:00

Target distribution:
stress_event_5pct_10d
0    2912
1    1984
Name: count, dtype: int64


In [4]:

# Verify the required columns and basic integrity
required_columns = ["date", "ticker", TARGET] + FEATURES
missing = set(required_columns) - set(data.columns)

if missing:
    raise ValueError(f"Missing columns: {sorted(missing)}")

if data.duplicated(["date", "ticker"]).any():
    raise ValueError("Duplicate (date, ticker) rows found.")

if not data[TARGET].isin([0, 1]).all():
    raise ValueError("Target must contain only 0 and 1.")

# Use common unique dates to define chronological boundaries
dates = pd.Index(sorted(data["date"].unique()))
n_dates = len(dates)

train_boundary = int(n_dates * 0.70)
validation_boundary = int(n_dates * 0.85)

# These dates define the end of training and validation
train_end_date = dates[train_boundary - 1]
validation_end_date = dates[validation_boundary - 1]

# Leave 10 trading-session dates between partitions.
# This is conservative for the 10-session forward target.
gap = 10

validation_start_date = dates[min(train_boundary + gap, n_dates - 1)]
test_start_date = dates[min(validation_boundary + gap, n_dates - 1)]

train = data[data["date"] <= train_end_date].copy()

validation = data[
    (data["date"] >= validation_start_date)
    & (data["date"] <= validation_end_date)
].copy()

test = data[data["date"] >= test_start_date].copy()

print("Training:", train["date"].min(), "to", train["date"].max())
print("Validation:", validation["date"].min(), "to", validation["date"].max())
print("Testing:", test["date"].min(), "to", test["date"].max())

print("\nRows:")
print("Train:", len(train))
print("Validation:", len(validation))
print("Test:", len(test))


Training: 2021-11-12 00:00:00 to 2025-04-11 00:00:00
Validation: 2025-04-29 00:00:00 to 2026-01-06 00:00:00
Testing: 2026-01-22 00:00:00 to 2026-09-30 00:00:00

Rows:
Train: 3424
Validation: 696
Test: 696


In [5]:

def summarize_split(name, frame):
    print(f"\n{name}")
    print("Rows:", len(frame))

    if frame.empty:
        print("WARNING: split is empty.")
        return

    print("Positive events:", int(frame[TARGET].sum()))
    print("Event rate (%):", round(frame[TARGET].mean() * 100, 2))

    print("\nBy stock:")
    print(
        frame.groupby("ticker")[TARGET]
        .agg(["count", "sum", "mean"])
        .assign(event_rate_pct=lambda x: x["mean"] * 100)
        .round(2)
    )

summarize_split("TRAIN", train)
summarize_split("VALIDATION", validation)
summarize_split("TEST", test)



TRAIN
Rows: 3424
Positive events: 1504
Event rate (%): 43.93

By stock:
        count  sum  mean  event_rate_pct
ticker                                  
AMD       856  436  0.51           50.93
INTC      856  388  0.45           45.33
NVDA      856  390  0.46           45.56
TSM       856  290  0.34           33.88

VALIDATION
Rows: 696
Positive events: 180
Event rate (%): 25.86

By stock:
        count  sum  mean  event_rate_pct
ticker                                  
AMD       174   70  0.40           40.23
INTC      174   53  0.30           30.46
NVDA      174   28  0.16           16.09
TSM       174   29  0.17           16.67

TEST
Rows: 696
Positive events: 281
Event rate (%): 40.37

By stock:
        count  sum  mean  event_rate_pct
ticker                                  
AMD       174   65  0.37           37.36
INTC      174   99  0.57           56.90
NVDA      174   69  0.40           39.66
TSM       174   48  0.28           27.59


In [6]:

for name, frame in [
    ("Train", train),
    ("Validation", validation),
    ("Test", test),
]:
    assert not frame.empty, f"{name} split is empty"
    assert frame[TARGET].nunique() == 2, (
        f"{name} does not contain both target classes"
    )
    assert frame[FEATURES].notna().all().all(), (
        f"{name} contains missing feature values"
    )
    assert np.isfinite(frame[FEATURES].to_numpy()).all(), (
        f"{name} contains infinite feature values"
    )

assert train["date"].max() < validation["date"].min()
assert validation["date"].max() < test["date"].min()

print("Basic split checks passed.")


Basic split checks passed.


In [7]:

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
)

majority_class = int(train[TARGET].mode()[0])

y_val = validation[TARGET]

baseline_predictions = np.full(
    len(validation),
    majority_class,
    dtype=int
)

print("Majority class:", majority_class)
print("Baseline accuracy:",
      accuracy_score(y_val, baseline_predictions))
print("Baseline precision:",
      precision_score(y_val, baseline_predictions, zero_division=0))
print("Baseline recall:",
      recall_score(y_val, baseline_predictions, zero_division=0))
print("Baseline F1:",
      f1_score(y_val, baseline_predictions, zero_division=0))


Majority class: 0
Baseline accuracy: 0.7413793103448276
Baseline precision: 0.0
Baseline recall: 0.0
Baseline F1: 0.0


In [8]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

X_train = train[FEATURES]
y_train = train[TARGET]

X_val = validation[FEATURES]
y_val = validation[TARGET]

model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    )),
])

model.fit(X_train, y_train)

val_probabilities = model.predict_proba(X_val)[:, 1]
val_predictions = (val_probabilities >= 0.5).astype(int)

In [9]:

from sklearn.metrics import confusion_matrix, classification_report

print("Accuracy:",
      accuracy_score(y_val, val_predictions))

print("Precision:",
      precision_score(y_val, val_predictions, zero_division=0))

print("Recall:",
      recall_score(y_val, val_predictions, zero_division=0))

print("F1-score:",
      f1_score(y_val, val_predictions, zero_division=0))

print("PR-AUC (Average Precision):",
      average_precision_score(y_val, val_probabilities))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, val_predictions))

print("\nClassification Report:")
print(classification_report(
    y_val,
    val_predictions,
    zero_division=0
))


Accuracy: 0.6551724137931034
Precision: 0.2222222222222222
Recall: 0.13333333333333333
F1-score: 0.16666666666666666
PR-AUC (Average Precision): 0.3013165071606081

Confusion Matrix:
[[432  84]
 [156  24]]

Classification Report:
              precision    recall  f1-score   support

           0       0.73      0.84      0.78       516
           1       0.22      0.13      0.17       180

    accuracy                           0.66       696
   macro avg       0.48      0.49      0.47       696
weighted avg       0.60      0.66      0.62       696

